# Tutorial 03 — Gasificador 0D: Barrido Paramétrico y Análisis de Sensibilidad

**Equipo:** Gasificador de lecho fijo · **Modo:** 0D (N=1) · Semibatch isobaro
**Prerrequisito:** Tutorial 02 — Gasificador 0D Semibatch

## Introducción

Este tutorial introduce las utilidades `parametric_sweep` y `sensitivity_analysis`
del módulo `src.utils.optimization` a través de un caso de referencia 0D.

### Objetivos

1. **Parametric sweep**: barrer T_wall sobre 8 valores, extraer métricas y comparar perfiles.
2. **Sensitivity analysis**: cuantificar cómo afecta cada parámetro al objetivo (conversión de biomasa).
3. **Timing serie vs paralelo**: medir el tiempo de 8 casos en serie, 2, 4 y 8 workers.

### Caso de referencia

Mismo reactor del Tutorial 02: semibatch 0D, modo isobaro (`v_out=None`), biomasa húmeda (mc_wb=15 %).
T_wall=1073 K (800 °C). Tiempo de simulación: 300 s (suficiente para capturar la pirólisis).

### Parámetros del benchmark

| Ajuste | Valor | Justificación |
|--------|-------|---------------|
| `T_MAX` | 300 s | La pirólisis activa ocurre en los primeros 300-600 s |
| `rtol` | 1e-4 | Más holgado que los tutoriales para reducir tiempos del benchmark |
| `n_sec` | 2 | Dos secciones de integración — suficiente para 0D |

### Paralelismo (joblib/loky — Windows)

La **primera llamada paralela** incluye el arranque de workers (~2-5 s).
Los workers se reutilizan entre llamadas siempre que el kernel esté activo.
Con casos de ~10-20 s/cada uno, la eficiencia paralela esperada es ≥ 80 %.

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.io.fuels_reader                    import read_fueldb
from src.units.gasifier.config.gas_props    import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props  import build_solid_prop_config
from src.units.gasifier.config.transport    import build_transport_config
from src.units.gasifier.config.thermal_bc   import build_thermal_bc_config
from src.units.gasifier.config.boundary_c   import build_bc_config
from src.units.gasifier.config.initial_c    import build_initial_c_config
from src.solvers.runner_gasifier            import run_step
from src.postprocessing.gasifier_balances   import check_balances, display_balances
from src.postprocessing.gasifier_plots      import (
    plot_sweep_profiles,
    plot_sweep_composition,
    plot_sweep_metrics,
)
from src.utils.optimization import parametric_sweep, sensitivity_analysis

FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")
SOLID_DB  = os.path.join(ROOT, "materials", "solids", "soliddb.txt")

fuel_config = read_fueldb(FUEL_PATH)
print(f"ROOT: {ROOT}")
print(f"Combustible: {fuel_config['description']}")
print(f"CPU cores disponibles: {os.cpu_count()}")

ROOT: C:\Users\MiguelCamaraSanz\OneDrive - Fundacion CIRCE\GITHUB\ProSimNet-gasifier
Combustible: Softwood (spruce) for fixed-bed updraft gasification
CPU cores disponibles: 12


## 1. Caso de referencia

In [2]:
# ── Geometría ─────────────────────────────────────────────────────────────────
N  = 1
L  = 0.10    # [m]  longitud del reactor
Di = 0.10    # [m]  diámetro interno
Do = 0.114   # [m]  diámetro externo
dz = L / N
Ai = 0.25 * np.pi * Di**2
Pi = np.pi * Di
Po = np.pi * Do
e_wall = (Do - Di) / 2

# ── Condiciones de operación ──────────────────────────────────────────────────
epsi_r      = 0.60        # [-]   porosidad del lecho
P_out       = 1.01325     # [bar] presión de referencia (isobaro)
T_wall_base = 1073.15     # [K]   800 °C — referencia del barrido

# ── Propiedades de pared ──────────────────────────────────────────────────────
k_wall   = 16.5     # [W/m/K]
rho_wall = 7950.0   # [kg/m³]
Cp_wall  = 510.0    # [J/kg/K]

# ── Combustible ───────────────────────────────────────────────────────────────
rho_p  = float(fuel_config["physical"]["rho_particle"])   # [kg/m³]
dp0    = float(fuel_config["physical"]["dp_initial"])     # [m]

# ── Fase sólida inicial ───────────────────────────────────────────────────────
mc_wb       = 0.15
rho_bio_dry = rho_p * (1 - epsi_r)
rho_moi_0   = rho_bio_dry * mc_wb / (1 - mc_wb)
rho_bio_0   = rho_bio_dry
rho_char_0  = 1e-6
rho_char0   = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)

# ── Propiedades del gas ───────────────────────────────────────────────────────
species      = list(GASIFIER_GAS_SPECIES)
nc           = len(species)
prop_gas     = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
gas_T_ref    = float(np.min(np.asarray(prop_gas["Tref"])))
MW_arr       = np.asarray(prop_gas["MW"])
solid_config = build_solid_prop_config(fuel_config)
trans_cfg    = build_transport_config(mode="constant", N=N, n_comp=nc,
                                      h_bed=80.0, h_wall=20.0)

# ── Condición inicial sv0 ─────────────────────────────────────────────────────
T_init = 300.0
y0     = np.zeros(nc)
y0[species.index("N2")] = 1.0

init_cfg = build_initial_c_config(
    P_init=P_out, Tg_init=T_init, Ts_init=T_init, y_init=y0,
    rho_biomass_init=rho_bio_0, rho_char_init=rho_char_0, rho_moisture_init=rho_moi_0,
    n_comp=nc, N=N, prop_gas=prop_gas, epsi_r=epsi_r, gas_T_ref=gas_T_ref, Tw_init=None,
)
sv0 = init_cfg["sv0"]

# ── params_base (sin bc_config ni thermal_bc_config: los añade cada run_fn) ───
params_base = {
    "n_comp": nc, "N": N, "dz": dz, "Ai": Ai, "Di": Di, "Pi": Pi, "Po": Po,
    "prop_gas": prop_gas, "MW": MW_arr, "gas_T_ref": gas_T_ref,
    "trans_config": trans_cfg, "energy": True,
    "epsi_r": epsi_r, "dp0": dp0, "rho_char0": rho_char0,
    "fuel_config": fuel_config, "solid_config": solid_config, "species": species,
}

# ── Parámetros del benchmark ──────────────────────────────────────────────────
T_MAX  = 300.0   # [s]  reducido para acelerar el benchmark
RTOL   = 1e-4    # tolerancias más holgadas (para benchmark)
ATOL   = 1e-6
N_SEC  = 2

print(f"sv0.shape = {sv0.shape}   ({sv0.shape[0]//N} DOF × {N} celda)")
print(f"T_MAX={T_MAX:.0f} s  rtol={RTOL}  atol={ATOL}  n_sec={N_SEC}")

sv0.shape = (17,)   (17 DOF × 1 celda)
T_MAX=300 s  rtol=0.0001  atol=1e-06  n_sec=2


## 2. Funciones de simulación y métricas

In [3]:
# ── run_fn para barrido de T_wall ─────────────────────────────────────────────
def run_twall(params):
    """Reconstruye thermal_bc_config con la T_wall del barrido."""
    Tw  = float(params["T_wall_K"])
    tbc = build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=Tw, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
    )
    bc  = build_bc_config(n_comp=nc, P_out_bar=P_out)   # v_out=None: isobaro
    p   = {**params_base, "bc_config": bc, "thermal_bc_config": tbc, "_cache": {}}
    t_arr, _, g = run_step(
        sv0=sv0, t_max=T_MAX, params=p, rtol=RTOL, atol=ATOL, n_sec=N_SEC,
        show_progress=bool(params.get("_show_progress", False)),
    )
    g._t = t_arr
    return g

# ── run_fn para sensitivity_analysis ─────────────────────────────────────────
def run_sens(params):
    """Usa thermal_bc_config ya parchado en params (por patcher_sens)."""
    bc = build_bc_config(n_comp=nc, P_out_bar=P_out)
    p  = {**params, "bc_config": bc, "_cache": {}}
    t_arr, _, g = run_step(
        sv0=sv0, t_max=T_MAX, params=p, rtol=RTOL, atol=ATOL, n_sec=N_SEC,
        show_progress=bool(params.get("_show_progress", False)),
    )
    g._t = t_arr
    return g

# ── patcher para sensitivity_analysis ────────────────────────────────────────
def patcher_sens(params, name, value):
    """Reconstruye thermal_bc_config si name='T_wall_K'; resto top-level."""
    p = {**params, "_cache": {}}
    if name == "T_wall_K":
        p["thermal_bc_config"] = build_thermal_bc_config(
            mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
            T_wall=float(value), k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
        )
    else:
        p[name] = value
    return p

# ── base_params para sensitivity (incluye bc_config y thermal_bc_config) ──────
base_sens = {
    **params_base,
    "bc_config": build_bc_config(n_comp=nc, P_out_bar=P_out),
    "thermal_bc_config": build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=T_wall_base, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
    ),
}

# ── métricas escalares comunes ────────────────────────────────────────────────
def metrics(g):
    """Métricas de fin de simulación."""
    return {
        "Ts_fin_C":     round(float(g._Ts_results[-1, 0])                       - 273.15, 1),
        "rho_bio_fin":  round(float(g._rho_solid_results[-1, 0, 0]),                       2),
        "rho_char_fin": round(float(g._rho_solid_results[-1, 1, 0]),                       2),
        "P_max_bar":    round(float(g._P_results.max()),                                    4),
        "conv_bio":     round(1.0 - float(g._rho_solid_results[-1, 0, 0]) / rho_bio_0,     3),
        "y_CO_fin":     round(float(g._y_results[-1, species.index("CO"),  0]),             4),
        "y_H2_fin":     round(float(g._y_results[-1, species.index("H2"),  0]),             4),
    }

print("Funciones listas.")

Funciones listas.


## 3. Calentamiento

Se ejecuta un caso en serie y dos casos con n_jobs=2 antes de medir tiempos.
Esto evita incluir el coste de importación de módulos y el arranque inicial
de los workers de joblib en las mediciones.

In [4]:
print("Warmup 1/2 — 1 caso serie...")
t0 = time.perf_counter()
_ = run_twall({"T_wall_K": T_wall_base})
t_por_caso = time.perf_counter() - t0
print(f"  tiempo por caso (referencia): {t_por_caso:.1f} s")

print("Warmup 2/2 — 2 casos paralelo (n_jobs=2, inicializa workers)...")
t0 = time.perf_counter()
parametric_sweep(
    base_params={},
    sweep_vars={"T_wall_K": [700.0, T_wall_base]},
    run_fn=run_twall,
    objective_fn=metrics,
    n_jobs=2, verbose=False, show_sim_progress=False,
)
t_warmup_par = time.perf_counter() - t0
print(f"  2 casos paralelo: {t_warmup_par:.1f} s")
print(f"  overhead de arranque estimado: ~{max(0.0, t_warmup_par - t_por_caso):.0f} s")
print(f"\nEstimado total del benchmark (4 configs × 8 casos): "
      f"~{(t_por_caso * 8 * 2.5):.0f} s")

Warmup 1/2 — 1 caso serie...
  tiempo por caso (referencia): 33.5 s
Warmup 2/2 — 2 casos paralelo (n_jobs=2, inicializa workers)...


Paralelo:   0%|          | 0/2 [00:00<?, ?caso/s]

Paralelo:   0%|          | 0/2 [00:00<?, ?caso/s]

  2 casos paralelo: 44.2 s
  overhead de arranque estimado: ~11 s

Estimado total del benchmark (4 configs × 8 casos): ~670 s


---
## 4. `parametric_sweep` — 8 casos de T_wall

Se barre T_wall ∈ {700, 750, 800, 850, 900, 950, 1000, 1073} K en modo isobaro
y se mide el tiempo para `n_jobs` ∈ {1, 2, 4, 8}.

In [ ]:
T_wall_sweep = [700.0, 750.0, 800.0, 850.0, 900.0, 950.0, 1000.0, 1073.15]
NJOBS_LIST   = [1, 2, 4, 8]

print(f"Barrido: {len(T_wall_sweep)} casos  |  Workers: {NJOBS_LIST}\n")

timing_sweep = {}
df_sweep_last = None
for n_jobs in NJOBS_LIST:
    t0 = time.perf_counter()
    df_s, res_s = parametric_sweep(
        base_params={},
        sweep_vars={"T_wall_K": T_wall_sweep},
        run_fn=run_twall,
        objective_fn=metrics,
        return_results=(n_jobs == NJOBS_LIST[-1]),   # guardar resultados solo en el ultimo
        n_jobs=n_jobs,
        verbose=False,
        show_sim_progress=False,
    )
    elapsed = time.perf_counter() - t0
    timing_sweep[n_jobs] = elapsed
    n_ok    = int((df_s["_status"] == "OK").sum())
    speedup = timing_sweep[1] / elapsed if n_jobs > 1 else 1.0
    print(f"  n_jobs={n_jobs:2d}:  {elapsed:6.1f} s   {n_ok}/{len(T_wall_sweep)} OK"
          f"   speedup={speedup:.2f}x")
    if n_jobs == NJOBS_LIST[-1]:
        df_sweep_last = df_s
        res_sweep_last = res_s

Barrido: 8 casos  |  Workers: [1, 2, 4, 8]



Barrido:   0%|          | 0/8 [00:00<?, ?caso/s]

### 4.1 Métricas del barrido

In [ ]:
display(df_sweep_last[["T_wall_K", "Ts_fin_C", "rho_bio_fin", "conv_bio",
                        "y_CO_fin", "y_H2_fin", "_status"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_sweep_profiles(df_sweep_last, res_sweep_last,
                    lambda g: g._Ts_results[:, 0],
                    "Ts [C]", "Temperatura del solido", "T_wall_K",
                    y_transform=lambda x: x - 273.15,
                    label_fn=lambda r: f"{r['T_wall_K']-273.15:.0f} C",
                    ax=axes[0])
plot_sweep_profiles(df_sweep_last, res_sweep_last,
                    lambda g: g._rho_solid_results[:, 0, 0],
                    "rho_biomasa [kg/m3_bed]", "Conversion de biomasa", "T_wall_K",
                    label_fn=lambda r: f"{r['T_wall_K']-273.15:.0f} C",
                    ax=axes[1])
plot_sweep_composition(df_sweep_last, res_sweep_last, "T_wall_K",
                       species_show=["CO", "CO2", "H2O", "H2", "CH4"],
                       label_fn=lambda r: f"{r['T_wall_K']-273.15:.0f} C",
                       ax=axes[2])
fig.suptitle("4 — Barrido T_wall: perfiles y composicion final", fontweight="bold")
fig.tight_layout(); plt.show()

---
## 5. `sensitivity_analysis` — 4 parámetros × ±10 %

Se perturban 4 parámetros físicos en ±10 % respecto al caso base.
Esto genera 4 × 2 = 8 casos de perturbación (+ 1 caso base siempre en serie).

| Parámetro | Valor base | Significado |
|-----------|-----------|-------------|
| `T_wall_K` | 1073.15 K | Temperatura de la pared exterior |
| `dp0` | 0.010 m | Diámetro inicial de partícula |
| `epsi_r` | 0.60 | Porosidad del lecho |
| `rho_char0` | calculado | Densidad de referencia del char |

In [ ]:
param_specs = {
    "T_wall_K":  T_wall_base,
    "dp0":       dp0,
    "epsi_r":    epsi_r,
    "rho_char0": rho_char0,
}
obj_conv = lambda g: 1.0 - float(g._rho_solid_results[-1, 0, 0]) / rho_bio_0

n_cases_sens = len(param_specs) * 2   # perturbaciones (sin contar el base)
print(f"Sensibilidad: {len(param_specs)} params × 2 = {n_cases_sens} perturbaciones"
      f" + 1 base = {n_cases_sens + 1} runs totales")
print(f"Workers: {NJOBS_LIST}\n")

timing_sens = {}
df_sens_last = None
for n_jobs in NJOBS_LIST:
    t0 = time.perf_counter()
    df_sa, res_sa = sensitivity_analysis(
        base_params=base_sens,
        param_specs=param_specs,
        run_fn=run_sens,
        objective_fn=obj_conv,
        param_patcher=patcher_sens,
        return_results=(n_jobs == NJOBS_LIST[-1]),
        n_jobs=n_jobs,
        verbose=False,
        show_sim_progress=False,
    )
    elapsed = time.perf_counter() - t0
    timing_sens[n_jobs] = elapsed
    speedup = timing_sens[1] / elapsed if n_jobs > 1 else 1.0
    print(f"  n_jobs={n_jobs:2d}:  {elapsed:6.1f} s   speedup={speedup:.2f}x")
    if n_jobs == NJOBS_LIST[-1]:
        df_sens_last = df_sa

### 5.1 Sensibilidad normalizada (objetivo: conversión de biomasa)

In [ ]:
display(df_sens_last[["v_base", "v_plus", "v_minus",
                       "f_plus", "f_minus", "S_plus", "S_minus", "S_mean"
                       ]].round(4).sort_values("S_mean", ascending=False))

fig, ax = plt.subplots(figsize=(7, 3.5))
df_plot = df_sens_last.sort_values("S_mean", ascending=True)
y_pos   = range(len(df_plot))
ax.barh(list(y_pos), df_plot["S_mean"], color="steelblue", alpha=0.85)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(df_plot.index.tolist())
ax.set(xlabel="|S_mean| (sensibilidad normalizada)", title="Sensibilidad de conv_bio")
ax.axvline(1.0, ls="--", color="gray", alpha=0.5, label="S=1 (proporcional)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="x")
fig.tight_layout(); plt.show()

---
## 6. Comparación de tiempos: serie vs paralelo

Se consolidan los tiempos de ambas utilidades y se calculan speedup y eficiencia.

In [ ]:
# ── DataFrame de tiempos ──────────────────────────────────────────────────────
df_timing = pd.DataFrame({
    "n_jobs":   NJOBS_LIST,
    "sweep_s":  [timing_sweep[n] for n in NJOBS_LIST],
    "sens_s":   [timing_sens[n]  for n in NJOBS_LIST],
})
df_timing["sweep_speedup"]    = df_timing["sweep_s"].iloc[0] / df_timing["sweep_s"]
df_timing["sens_speedup"]     = df_timing["sens_s"].iloc[0]  / df_timing["sens_s"]
df_timing["sweep_efficiency"] = df_timing["sweep_speedup"] / df_timing["n_jobs"] * 100
df_timing["sens_efficiency"]  = df_timing["sens_speedup"]  / df_timing["n_jobs"] * 100
df_timing = df_timing.set_index("n_jobs")
display(df_timing.round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Tiempo de ejecucion
x = np.arange(len(NJOBS_LIST))
w = 0.35
axes[0].bar(x - w/2, df_timing["sweep_s"], w,
            label=f"parametric_sweep ({len(T_wall_sweep)} casos)",
            color="steelblue", alpha=0.85)
axes[0].bar(x + w/2, df_timing["sens_s"], w,
            label=f"sensitivity_analysis ({n_cases_sens} casos + 1 base)",
            color="darkorange", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"n_jobs={n}" for n in NJOBS_LIST], fontsize=9)
axes[0].set(ylabel="Tiempo [s]", title="Tiempo de ejecucion")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Speedup
ideal = np.linspace(1, max(NJOBS_LIST), 100)
axes[1].plot(NJOBS_LIST, df_timing["sweep_speedup"], "o-", color="steelblue",
             lw=2, ms=7, label="parametric_sweep")
axes[1].plot(NJOBS_LIST, df_timing["sens_speedup"],  "s-", color="darkorange",
             lw=2, ms=7, label="sensitivity_analysis")
axes[1].plot(ideal, ideal, "--", color="gray", alpha=0.5, label="ideal lineal")
axes[1].set(xlabel="n_jobs", ylabel="Speedup [-]", title="Speedup vs n_jobs")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Eficiencia paralela
axes[2].plot(NJOBS_LIST, df_timing["sweep_efficiency"], "o-", color="steelblue",
             lw=2, ms=7, label="parametric_sweep")
axes[2].plot(NJOBS_LIST, df_timing["sens_efficiency"],  "s-", color="darkorange",
             lw=2, ms=7, label="sensitivity_analysis")
axes[2].axhline(100, ls="--", color="gray", alpha=0.5, label="ideal (100 %)")
axes[2].set_ylim(0, 120)
axes[2].set(xlabel="n_jobs", ylabel="Eficiencia [%]", title="Eficiencia del paralelismo")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

fig.suptitle("6 — Serie vs Paralelo: speedup y eficiencia", fontweight="bold")
fig.tight_layout(); plt.show()

## Interpretación de los resultados

### Tabla de eficiencia esperada

| Eficiencia | Significado |
|-----------|-------------|
| > 90 % | Paralelismo excelente — el cómputo domina sobre el overhead |
| 70–90 % | Bueno — overhead de serialización y arranque visible pero menor |
| 50–70 % | Aceptable — revisar si el tiempo por caso es suficientemente largo |
| < 50 % | Overhead domina — los casos son demasiado rápidos para paralelizar |

### `sensitivity_analysis` vs `parametric_sweep`

Ambas utilidades incluyen **todos sus casos en el mismo lote paralelo**,
incluido el caso base de `sensitivity_analysis`. El speedup teórico es el mismo:

```
speedup(n_jobs) ≈ n_jobs × eficiencia(n_jobs)
```

Si `sensitivity_analysis` muestra menor speedup que `parametric_sweep`,
se debe únicamente a que tiene más casos totales (N_params×2 + 1) lo que puede
cambiar el balance carga/overhead según la máquina.

### Cuándo usar `n_jobs > 1`

- Cada simulación dura **> 10 s**: rentable desde n_jobs=2.
- Cada simulación dura **1–10 s**: rentable desde n_jobs=4 en adelante.
- Cada simulación dura **< 1 s**: el overhead de workers supera el beneficio → usar serie.

---
## 7. Verificación de balances del caso base

Se re-ejecuta el caso base con tolerancias ajustadas para verificar el cierre.

In [ ]:
tbc_ref = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_base, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
)
bc_ref  = build_bc_config(n_comp=nc, P_out_bar=P_out)
p_ref   = {**params_base, "bc_config": bc_ref, "thermal_bc_config": tbc_ref}

print("Ejecutando caso base con tolerancias ajustadas para verificacion de balances...")
t_ref, _, g_ref = run_step(sv0=sv0, t_max=T_MAX, params=p_ref,
                            rtol=1e-5, atol=1e-7, n_sec=5, show_progress=True)
bal = check_balances(g_ref, p_ref, verbose=False)
display_balances(bal)

## Conclusiones

### `parametric_sweep`

- Barrido cartesiano sobre cualquier combinación de parámetros.
- La `run_fn` reconstruye solo lo que cambia por caso (e.g. `thermal_bc_config`).
- Con `return_results=True` se conservan los perfiles completos para análisis posterior.
- `n_jobs=-1` usa todos los cores disponibles; recomendado cuando hay ≥ 4 casos y
  cada uno tarda > 10 s.

### `sensitivity_analysis`

- Perturba cada parámetro ±`delta_pct` (10 % por defecto) respecto al caso base.
- Calcula la sensibilidad normalizada: `S = (Δf/f_base) / (Δp/p_base)`.
- `|S| > 1`: el objetivo es más sensible al parámetro que proporcional.
- El caso base se ejecuta siempre en serie — el speedup máximo es limitado por esta constante.

### Qué explorar a continuación (Tutorial 04)

- Señales variables en las condiciones de contorno (ramp, step, pulse, feedforward).
- Control proporcional de T_wall por retroalimentación del estado.